In [11]:
using LowLevelFEM, LinearAlgebra
using BenchmarkTools

structured_box_mesh(n=30)
mat = Material("body");

In [12]:
P1 = Problem([mat])

u = @btime begin
    bc = displacementConstraint("left", ux=0, uy=0, uz=0)
    ld = load("right", fx=1)
    solveDisplacement($P1, load=[ld], support=[bc])
end

@btime S = solveStress($u);

  62.723 s (15091521 allocations: 12.72 GiB)
  1.201 s (16956257 allocations: 1.24 GiB)


In [13]:
P2 = Problem([mat], type=:VectorField, dim=3, field=:u)

u = @btime begin
    μ = $mat.μ
    λ = $mat.λ
    D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

    K = ∫(SymGrad($P2) ⋅ D ⋅ SymGrad($P2))

    f = ∫($P2 ⋅ [1.0, 0.0, 0.0], Γ="right")

    bc = BoundaryCondition("left", ux=0, uy=0, uz=0)

    solveField(Symmetric(K), f, support=[bc])
    #solveField(K, f, support=[bc])
end

@btime begin
    A = ($u ∘ ∇ + ∇ ∘ $u) / 2.0

    I = TensorField($P2, "body", [1 0 0; 0 1 0; 0 0 1])
    E = $mat.E
    ν = $mat.ν

    S = E / (1 + ν) * (A + ν / (1 - 2ν) * trace(A) * I)
end;

  48.130 s (286822 allocations: 3.17 GiB)
  1.064 s (3133465 allocations: 356.62 MiB)
